In [ ]:
!pip install python-docx -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve
from imblearn.over_sampling import SMOTE
from docx import Document
from docx.shared import Inches
import time
import os
import gc

In [ ]:
doc = Document()
doc.add_heading('Fraud Detection Model Analysis Results', 0)

sample_frac = 0.1
base_file = 'Base.csv'
variant_files = ['Variant I.csv', 'Variant II.csv', 'Variant III.csv', 'Variant IV.csv', 'Variant V.csv']

def add_plot(doc, fname, title):
    plt.savefig(fname, bbox_inches='tight', dpi=300)
    doc.add_heading(title, level=2)
    doc.add_picture(fname, width=Inches(6.0))
    os.remove(fname)
    plt.clf()

In [ ]:
df = pd.read_csv(base_file).sample(frac=sample_frac, random_state=42)

drop_c = ['x1', 'x2']
df = df.drop(columns=[c for c in drop_c if c in df.columns])

X = df.drop(columns=['fraud_bool'])
y = df['fraud_bool']

X_enc = pd.get_dummies(X)
train_cols = X_enc.columns

X_tr, X_te, y_tr, y_te = train_test_split(X_enc, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=y_tr)
add_plot(doc, 'pre.png', 'Class Imbalance Before SMOTE')

sm = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)

plt.figure(figsize=(6,4))
sns.countplot(x=y_tr_sm)
add_plot(doc, 'post.png', 'Class Imbalance After SMOTE')

del df, X, X_enc, X_tr, y_tr
gc.collect()

In [ ]:
models = {
    'RF': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGB': XGBClassifier(random_state=42, eval_metric='logloss')
}

est = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=50, random_state=42, eval_metric='logloss'))
]

models['Ensemble'] = StackingClassifier(
    estimators=est, 
    final_estimator=LogisticRegression(),
    n_jobs=-1
)

In [ ]:
res = []
trained = {}

for name, clf in models.items():
    t0 = time.time()
    clf.fit(X_tr_sm, y_tr_sm)
    t_train = time.time() - t0
    
    t1 = time.time()
    preds = clf.predict(X_te)
    probs = clf.predict_proba(X_te)[:, 1]
    t_inf = time.time() - t1
    
    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    
    res.append({
        'Model': name,
        'Precision': precision_score(y_te, preds),
        'Recall': recall_score(y_te, preds),
        'F1-Score': f1_score(y_te, preds),
        'AUC-ROC': roc_auc_score(y_te, probs),
        'FPR': fp / (fp + tn),
        'Train Time': round(t_train, 2),
        'Infer Time': round(t_inf, 2)
    })
    trained[name] = clf

res_df = pd.DataFrame(res)

doc.add_heading('Base Dataset Metrics', level=2)
table = doc.add_table(rows=1, cols=len(res_df.columns))
table.style = 'Table Grid'
cells = table.rows[0].cells
for i, col in enumerate(res_df.columns):
    cells[i].text = col
for _, row in res_df.iterrows():
    r_cells = table.add_row().cells
    for i, val in enumerate(row):
        r_cells[i].text = str(round(val, 4)) if isinstance(val, float) else str(val)

In [ ]:
preds_ens = trained['Ensemble'].predict(X_te)
cm = confusion_matrix(y_te, preds_ens)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
add_plot(doc, 'cm.png', 'Confusion Matrix')

plt.figure(figsize=(8,5))
for name, clf in trained.items():
    probs = clf.predict_proba(X_te)[:, 1]
    prec, rec, _ = precision_recall_curve(y_te, probs)
    plt.plot(rec, prec, label=name)
plt.legend()
add_plot(doc, 'pr.png', 'PR Curve')

del X_tr_sm, y_tr_sm, X_te, y_te
gc.collect()

In [ ]:
doc.add_heading('Variant Tests', level=1)
ens = trained['Ensemble']

for v in variant_files:
    if not os.path.exists(v):
        continue
        
    df_v = pd.read_csv(v).sample(frac=sample_frac, random_state=42)
    df_v = df_v.drop(columns=[c for c in drop_c if c in df_v.columns])
    
    X_v = df_v.drop(columns=['fraud_bool'])
    y_v = df_v['fraud_bool']
    
    X_v_enc = pd.get_dummies(X_v).reindex(columns=train_cols, fill_value=0)
    
    p_v = ens.predict(X_v_enc)
    prob_v = ens.predict_proba(X_v_enc)[:, 1]
    
    tn, fp, fn, tp = confusion_matrix(y_v, p_v).ravel()
    
    doc.add_heading(v, level=2)
    doc.add_paragraph(f"F1: {f1_score(y_v, p_v):.4f}")
    doc.add_paragraph(f"AUC: {roc_auc_score(y_v, prob_v):.4f}")
    doc.add_paragraph(f"Precision: {precision_score(y_v, p_v):.4f}")
    doc.add_paragraph(f"FPR: {fp / (fp + tn):.4f}")
    
    del df_v, X_v, X_v_enc, y_v
    gc.collect()

doc.save('results.docx')